# Coupled stress-expert optimization

This inverse-design workflow trains ensemble surrogates for mass, payload, and fuel, plus coupled stress experts. It calibrates a conservative stress upper bound, then uses differential evolution to minimize the competition loss while satisfying the stress constraint.

Results are written as a CSV submission summary plus candidate and diagnostic JSON files under `outputs/`. Structural validation is surrogate-only because this repository does not expose a callable FE solver.

The database fallback is used only if no optimized design passes the conservative stress screen; database geometries are re-evaluated for the requested mission before selection.


## Run this notebook

Run all cells from the repository root. Install dependencies first with `pip install -r requirements.txt`. The notebook writes `coupled_stress_output_summary.csv`, `coupled_stress_candidates.json`, `coupled_stress_diagnostics.json`, and `coupled_stress_validation.png` to `outputs/`.

In [20]:
from __future__ import annotations

# If these imports fail, run: pip install -r requirements.txt

import hashlib
import json
import os
from pathlib import Path
import sys

os.environ.setdefault('MPLCONFIGDIR', '/tmp/bwb_coupled_stress_matplotlib')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import mean_absolute_error, roc_auc_score
from sklearn.model_selection import train_test_split



In [21]:
# Hyperparameters for this code
RNG_SEED = 20260819
N_RESTARTS = 3
DE_GENERATIONS = 50
DE_POPSIZE = 8
PERTURB_TRIALS = 3
PERTURB_GENERATIONS = 30
PERTURB_RADIUS = 0.05
MOVE_TOL = 0.01
LOSS_TOL = 1e-4
CONSTRAINT_WEIGHT = 1e6
GATE_THRESHOLD = 345.0


In [22]:
#Defaults and Path information for Later Functions
ROOT = Path.cwd()
if not (ROOT / 'data' / 'bwb_structures_dataset.csv').exists():
    raise FileNotFoundError('Run this notebook from the repository root.')
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / 'models' / 'ld_surrogate'))
from predict_ld import predict_ld_batch

DESIGN_COLUMNS = [
    'C2/C1', 'C3/C1', 'C4/C1', 'B1/C1', 'B2/C1', 'B3/C1', 'X3/C1', 'S1', 'S3', 'C1',
    'Skin Thickness', 'Front Spar Chord %', 'Rear Spar Chord %', 'Spar Thickness', '# of Ribs',
    'Rib Thickness', 'Wingbox Cutout', '# of Fuselage Ribs', '# of Fuselage Spars',
    'Fuselage Struct Thickness', 'Fuselage Struct Width',
]
FLIGHT_COLUMNS = ['Altitude', 'KCAS', 'AOA']
FEATURE_COLUMNS = DESIGN_COLUMNS + FLIGHT_COLUMNS
TARGET_COLUMNS = ['Aircraft Empty Weight', 'Payload Volume', 'Fuel Volume', 'Max Hotspot Stress']
BOUNDS = np.array([
    [.55, .85], [.18, .28], [.06, .09], [.1, .2], [.05, .2], [.35, .7], [.5, .65], [40, 60], [20, 40],
    [2500, 4000], [.0003, .005], [.18, .35], [.55, .75], [.00098, .008], [3, 14], [.0015, .015],
    [.01, .05], [3, 11], [3, 12], [.002, .025], [.001, .015],
])
LO, HI = BOUNDS[:, 0], BOUNDS[:, 1]
SPAN = HI - LO

def normalize(designs):
    return (np.asarray(designs, dtype=float) - LO) / SPAN

def repair_design(designs):
    repaired = np.clip(np.asarray(designs, dtype=float).copy(), LO, HI)
    repaired[..., 14] = np.rint(repaired[..., 14])
    repaired[..., 17] = 2 * np.rint((repaired[..., 17] - 3) / 2) + 3
    repaired[..., 18] = np.rint(repaired[..., 18])
    return repaired

def mission_for_model(mission, name):
    return {
        'mission': name, 'altitude': float(mission['altitude_kft']), 'kcas': float(mission['kcas_kt']),
        'aoa': float(mission['aoa_deg']), 'ld_target': float(mission['ld_target']),
        'payload_target': float(mission['payload_volume_min_m3']),
        'fuel_target': float(mission['fuel_volume_min_m3']),
        'stress_limit': float(mission['stress_max_mpa']),
    }

data = pd.read_csv(ROOT / 'data' / 'bwb_structures_dataset.csv')
data = data.loc[data['Max Hotspot Stress'] < 1e4].reset_index(drop=True)
designs_db = data[DESIGN_COLUMNS].to_numpy(float)
features = data[FEATURE_COLUMNS].to_numpy(float)
outputs = data[TARGET_COLUMNS].to_numpy(float)
outputs[:, 1:3] /= 1e9
stress = outputs[:, 3]
stress_bands = np.digitize(stress, [250, 300, 320, 335, 355])
all_rows = np.arange(len(data))
train_cal, test_idx = train_test_split(all_rows, test_size=.15, random_state=RNG_SEED, stratify=stress_bands)
train_idx, cal_idx = train_test_split(
    train_cal, test_size=.15 / .85, random_state=RNG_SEED + 1, stratify=stress_bands[train_cal],
)
print(f'Clean rows: {len(data):,}; train={len(train_idx):,}, calibration={len(cal_idx):,}, test={len(test_idx):,}')

Clean rows: 13,597; train=9,517, calibration=2,040, test=2,040


### Ensemble surrogates

Mass, payload, and fuel each use a five-member bootstrap ensemble. Three members are ExtraTrees regressors, which capture nonlinear feature interactions through randomized decision trees; two are histogram gradient-boosting regressors, which provide a complementary tree-based fit. The ensemble mean is the prediction and the spread across members is used as an uncertainty estimate.

Each member is trained on a resampled training set, so the model can estimate how sensitive its prediction is to the available data.

In [23]:
class BootstrapEnsemble:
    def __init__(self, members, inverse=None):
        self.members = members
        self.inverse = inverse

    def member_predictions(self, x):
        values = np.vstack([model.predict(x) for model in self.members])
        return self.inverse(values) if self.inverse is not None else values

    def predict(self, x):
        values = self.member_predictions(x)
        return values.mean(axis=0), values.std(axis=0, ddof=1), values

def gaussian_weight(y, center, width):
    return np.exp(-.5 * ((np.asarray(y) - center) / width) ** 2)

def fit_ensemble(x, y, seed, sample_weight=None, inverse=None):
    rng = np.random.default_rng(seed)
    probability = np.ones(len(x)) if sample_weight is None else np.asarray(sample_weight, dtype=float)
    probability = probability / probability.sum()
    members = []
    for member in range(5):
        sampled = rng.choice(len(x), size=len(x), replace=True, p=probability)
        if member < 3:
            model = ExtraTreesRegressor(
                n_estimators=64, min_samples_leaf=3, max_features=.85, bootstrap=True,
                max_samples=.9, n_jobs=1, random_state=seed + member,
            )
        else:
            model = HistGradientBoostingRegressor(
                learning_rate=.06, max_iter=160, max_leaf_nodes=31, min_samples_leaf=18,
                l2_regularization=.25, random_state=seed + member,
            )
        model.fit(x[sampled], y[sampled])
        members.append(model)
    return BootstrapEnsemble(members, inverse=inverse)

global_models = []
for target in range(3):
    global_models.append(fit_ensemble(features[train_idx], outputs[train_idx, target], RNG_SEED + 100 * target))

train_stress = stress[train_idx]
low_keep = train_stress <= 450.0
high_keep = train_stress >= 250.0
low_weight = 1.0 + 3.0 * gaussian_weight(train_stress[low_keep], 335.0, 35.0) + 2.0 * gaussian_weight(train_stress[low_keep], 355.0, 35.0)
high_weight = 1.0 + 3.0 * gaussian_weight(train_stress[high_keep], 335.0, 30.0) + 3.0 * gaussian_weight(train_stress[high_keep], 355.0, 30.0)
stress_low = fit_ensemble(
    features[train_idx][low_keep], np.log1p(train_stress[low_keep]), RNG_SEED + 400,
    sample_weight=low_weight, inverse=np.expm1,
)
stress_high = fit_ensemble(
    features[train_idx][high_keep], np.log1p(train_stress[high_keep]), RNG_SEED + 500,
    sample_weight=high_weight, inverse=np.expm1,
)
gate = ExtraTreesClassifier(
    n_estimators=256, min_samples_leaf=4, max_features=.85, bootstrap=True, max_samples=.9,
    class_weight='balanced', n_jobs=1, random_state=RNG_SEED + 600,
)
gate.fit(features[train_idx], train_stress >= GATE_THRESHOLD)
raw_gate_cal = gate.predict_proba(features[cal_idx])[:, 1]
gate_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')
gate_calibrator.fit(raw_gate_cal, (stress[cal_idx] >= GATE_THRESHOLD).astype(float))
print('Fitted 15 global-response members, 10 coupled stress-expert members, and one calibrated gate.')

Fitted 15 global-response members, 10 coupled stress-expert members, and one calibrated gate.


### Coupled stress prediction

Stress uses two overlapping five-member ensembles: a low-stress expert trained through 450 MPa and a high-stress expert trained from 250 MPa upward. Both emphasize samples near 335 and 355 MPa, where the optimizer needs reliable constraint decisions.

A calibrated ExtraTrees classifier estimates the probability that a design is above the 345 MPa gate. That probability blends the two experts; their within-model uncertainty and disagreement are combined. A one-sided bound, calibrated on held-out data, is the value compared with the mission stress limit.

In [24]:
def coupled_stress_prediction(x):
    low_mu, low_sd, _ = stress_low.predict(x)
    high_mu, high_sd, _ = stress_high.predict(x)
    raw_probability = gate.predict_proba(x)[:, 1]
    high_probability = np.clip(gate_calibrator.predict(raw_probability), .02, .98)
    mean = (1.0 - high_probability) * low_mu + high_probability * high_mu
    variance = (
        (1.0 - high_probability) * (low_sd ** 2 + (low_mu - mean) ** 2)
        + high_probability * (high_sd ** 2 + (high_mu - mean) ** 2)
    )
    return mean, np.sqrt(np.maximum(variance, 1e-12)), high_probability, low_mu, high_mu

cal_stress_mu, cal_stress_sd, _, _, _ = coupled_stress_prediction(features[cal_idx])
stress_scale_floor = float(max(5.0, np.quantile(cal_stress_sd, .25)))
cal_scale = np.maximum(cal_stress_sd, stress_scale_floor)
one_sided_ratio = (stress[cal_idx] - cal_stress_mu) / cal_scale
boundary_cal = (stress[cal_idx] >= 250.0) & (stress[cal_idx] <= 450.0)
stress_k = float(max(0.0, np.quantile(one_sided_ratio, .90), np.quantile(one_sided_ratio[boundary_cal], .90)))

def stress_upper_bound(mean, std):
    return np.asarray(mean) + stress_k * np.maximum(np.asarray(std), stress_scale_floor)

def structural_prediction(x):
    means, stds = [], []
    for model in global_models:
        mean, std, _ = model.predict(x)
        means.append(np.maximum(mean, 0.0))
        stds.append(std)
    stress_mu, stress_sd, gate_probability, low_mu, high_mu = coupled_stress_prediction(x)
    mean = np.column_stack(means + [np.maximum(stress_mu, 0.0)])
    std = np.column_stack(stds + [stress_sd])
    return mean, std, gate_probability, low_mu, high_mu

def feature_matrix(candidate_designs, mission):
    repaired = repair_design(np.atleast_2d(candidate_designs))
    flight = np.array([mission['altitude'], mission['kcas'], mission['aoa']], dtype=float)
    x = np.column_stack([repaired, np.broadcast_to(flight, (len(repaired), 3))])
    return repaired, x

def official_loss(ld, values, mission):
    return (
        .4 * values[:, 0] / 50.0
        + .2 * np.maximum(0.0, (mission['ld_target'] - ld) / mission['ld_target'])
        + .2 * np.maximum(0.0, (mission['payload_target'] - values[:, 1]) / mission['payload_target'])
        + .2 * np.maximum(0.0, (mission['fuel_target'] - values[:, 2]) / mission['fuel_target'])
    )

def evaluate_designs(candidate_designs, mission):
    repaired, x = feature_matrix(candidate_designs, mission)
    mean, std, gate_probability, low_mu, high_mu = structural_prediction(x)
    frame = pd.DataFrame(repaired, columns=DESIGN_COLUMNS)
    ld = predict_ld_batch(frame, alt_kft=mission['altitude'], kcas=mission['kcas'], aoa=mission['aoa'])
    loss = official_loss(ld, mean, mission)
    ucb = stress_upper_bound(mean[:, 3], std[:, 3])
    return {
        'designs': repaired, 'ld': ld, 'mean': mean, 'std': std, 'gate_probability': gate_probability,
        'low_stress': low_mu, 'high_stress': high_mu, 'stress_ucb': ucb, 'loss': loss,
    }

test_mu, test_sd, test_gate, test_low, test_high = structural_prediction(features[test_idx])
test_ucb = stress_upper_bound(test_mu[:, 3], test_sd[:, 3])
band_edges = [-np.inf, 250, 300, 320, 335, 355, np.inf]
band_names = ['<250', '250-300', '300-320', '320-335', '335-355', '>=355']
validation_rows = []
for name, lower, upper in zip(band_names, band_edges[:-1], band_edges[1:]):
    mask = (stress[test_idx] >= lower) & (stress[test_idx] < upper)
    validation_rows.append({
        'band': name, 'rows': int(mask.sum()),
        'mae_mpa': float(mean_absolute_error(stress[test_idx][mask], test_mu[mask, 3])),
        'bias_mpa': float(np.mean(test_mu[mask, 3] - stress[test_idx][mask])),
        'ucb_coverage': float(np.mean(stress[test_idx][mask] <= test_ucb[mask])),
    })
validation = pd.DataFrame(validation_rows)
print(validation.to_string(index=False))
print(f'Overall stress MAE={mean_absolute_error(stress[test_idx], test_mu[:, 3]):.2f} MPa; '
      f'UCB coverage={np.mean(stress[test_idx] <= test_ucb):.3f}; k={stress_k:.3f}; floor={stress_scale_floor:.2f} MPa; '
      f'gate AUC={roc_auc_score(stress[test_idx] >= GATE_THRESHOLD, test_gate):.3f}')

   band  rows    mae_mpa    bias_mpa  ucb_coverage
   <250  1071  64.348927   49.390425      1.000000
250-300    94 109.465155   38.409680      1.000000
300-320    31 123.977466   23.584281      1.000000
320-335    21 125.058500   41.363382      1.000000
335-355    31 132.372690   11.330594      1.000000
  >=355   792 529.777521 -395.579391      0.784091
Overall stress MAE=249.69 MPa; UCB coverage=0.916; k=3.855; floor=75.80 MPa; gate AUC=0.932


### Optimization and conservative selection

Each mission starts with three global differential-evolution searches in normalized design space. Short local perturbation searches then check whether the best solution is stable. Candidate designs are ranked by official loss only after passing the calibrated stress upper-bound screen; if none pass, the notebook falls back to a mission-rescored database design.

In [25]:
def stable_seed(case_id, offset=0):
    digest = hashlib.sha256(case_id.encode('utf-8')).digest()
    return RNG_SEED + int.from_bytes(digest[:4], 'big') % 1_000_000 + offset

def penalty_score(evaluation, mission):
    violation = np.maximum(0.0, (evaluation['stress_ucb'] - mission['stress_limit']) / 20.0)
    return evaluation['loss'] + CONSTRAINT_WEIGHT * violation ** 2

def run_de(mission, seed, generations, label, local_bounds=None, x0=None):
    evaluated = []
    history = []
    previous = None
    bounds = [(0.0, 1.0)] * len(DESIGN_COLUMNS) if local_bounds is None else local_bounds

    def objective(z):
        array = np.asarray(z, dtype=float)
        scalar = array.ndim == 1
        if scalar:
            normalized_batch = array[None, :]
        elif array.shape[0] == len(DESIGN_COLUMNS):
            normalized_batch = array.T
        else:
            normalized_batch = array
        candidate_batch = repair_design(LO + normalized_batch * SPAN)
        evaluated.append(candidate_batch.copy())
        score = penalty_score(evaluate_designs(candidate_batch, mission), mission)
        return float(score[0]) if scalar else score

    def callback(xk, convergence):
        nonlocal previous
        current = repair_design(LO + np.asarray(xk) * SPAN)[None, :]
        current_norm = normalize(current)[0]
        movement = None if previous is None else float(np.max(np.abs(current_norm - previous)))
        current_eval = evaluate_designs(current, mission)
        history.append({
            'generation': len(history) + 1, 'movement': movement, 'convergence': float(convergence),
            'loss': float(current_eval['loss'][0]), 'stress_mean': float(current_eval['mean'][0, 3]),
            'stress_ucb': float(current_eval['stress_ucb'][0]),
        })
        previous = current_norm
        return False

    safe_x0 = None
    if x0 is not None:
        bounds_array = np.asarray(bounds, dtype=float)
        safe_x0 = np.clip(np.asarray(x0, dtype=float), bounds_array[:, 0], bounds_array[:, 1])
    result = differential_evolution(
        objective, bounds, maxiter=generations, popsize=DE_POPSIZE, init='sobol',
        rng=seed, callback=callback, polish=False, updating='deferred', workers=1, vectorized=True,
        tol=1e-4, atol=0.0, x0=safe_x0,
    )
    design = repair_design((LO + result.x * SPAN)[None, :])[0]
    return {
        'label': label, 'seed': seed, 'result': result, 'design': design, 'normalized': normalize(design),
        'score': float(result.fun), 'history': history, 'evaluated': evaluated,
    }

def optimize_mission(mission, case_id):
    searches = []
    evaluated_batches = []
    for restart in range(N_RESTARTS):
        run = run_de(mission, stable_seed(case_id, restart), DE_GENERATIONS, f'global_{restart + 1}')
        searches.append(run)
        evaluated_batches.extend(run['evaluated'])

    anchor = min(searches, key=lambda item: item['score'])
    perturbation_cycles = []
    for cycle in range(2):
        center = np.clip(anchor['normalized'], 0.0, 1.0)
        local_bounds = list(zip(np.maximum(0.0, center - PERTURB_RADIUS), np.minimum(1.0, center + PERTURB_RADIUS)))
        trials = []
        for trial in range(PERTURB_TRIALS):
            run = run_de(
                mission, stable_seed(case_id, 100 + cycle * 10 + trial), PERTURB_GENERATIONS,
                f'perturb_{cycle + 1}_{trial + 1}', local_bounds=local_bounds, x0=center,
            )
            run['return_distance'] = float(np.max(np.abs(run['normalized'] - center)))
            run['loss_delta'] = float(evaluate_designs(run['design'][None, :], mission)['loss'][0] - evaluate_designs(anchor['design'][None, :], mission)['loss'][0])
            run['stable'] = bool(run['return_distance'] <= MOVE_TOL and abs(run['loss_delta']) <= LOSS_TOL)
            trials.append(run)
            evaluated_batches.extend(run['evaluated'])
        best_trial = min(trials, key=lambda item: item['score'])
        improved = best_trial['score'] < anchor['score'] - LOSS_TOL
        perturbation_cycles.append({'cycle': cycle + 1, 'improved': bool(improved), 'trials': trials})
        if cycle == 0 and improved:
            anchor = best_trial
            continue
        break

    pool = repair_design(np.vstack(evaluated_batches + [anchor['design'][None, :]]))
    rounded = np.round(pool, 10)
    _, unique_idx = np.unique(rounded, axis=0, return_index=True)
    pool = pool[np.sort(unique_idx)]
    pool_eval = evaluate_designs(pool, mission)
    conservative = pool_eval['stress_ucb'] <= mission['stress_limit']
    fallback_row = None
    if conservative.any():
        eligible = np.flatnonzero(conservative)
        selected = int(eligible[np.argmin(pool_eval['loss'][eligible])])
        chosen_eval = {key: value[[selected]] if isinstance(value, np.ndarray) else value for key, value in pool_eval.items()}
        source = 'optimized_conservative'
    else:
        database_eval = evaluate_designs(designs_db, mission)
        mean_safe = database_eval['mean'][:, 3] <= mission['stress_limit']
        if mean_safe.any():
            eligible = np.flatnonzero(mean_safe)
            closeness = np.abs(mission['stress_limit'] - database_eval['mean'][eligible, 3])
            selected = int(eligible[np.lexsort((closeness, database_eval['loss'][eligible]))[0]])
            source = 'database_mean_safe_fallback'
        else:
            selected = int(np.argmin(database_eval['mean'][:, 3]))
            source = 'no_mean_safe_candidate'
        fallback_row = selected
        chosen_eval = {key: value[[selected]] if isinstance(value, np.ndarray) else value for key, value in database_eval.items()}

    histories = [{key: value for key, value in run.items() if key in ('label', 'seed', 'score', 'history')} for run in searches]
    perturbations = []
    for cycle in perturbation_cycles:
        perturbations.append({
            'cycle': cycle['cycle'], 'improved': cycle['improved'],
            'trials': [
                {'label': trial['label'], 'score': trial['score'], 'return_distance': trial['return_distance'],
                 'loss_delta': trial['loss_delta'], 'stable': trial['stable']} for trial in cycle['trials']
            ],
        })
    return chosen_eval, {
        'source': source, 'fallback_dataset_row': fallback_row, 'pool_size': int(len(pool)),
        'conservative_pool_count': int(conservative.sum()), 'global_searches': histories,
        'perturbation_cycles': perturbations,
    }

### Generate candidate files

For every runnable case, the selected design, mission conditions, and predicted metrics are saved to a CSV summary and the candidate JSON. The diagnostic JSON preserves the stress uncertainty, selection path, validation summary, and search history needed to interpret that selection.

In [26]:
cases_payload = json.loads((ROOT / 'gauntlet' / 'runnable_cases.json').read_text(encoding='utf-8'))
candidate_records = []
diagnostic_records = []
for case in cases_payload['cases']:
    mission = mission_for_model(case['mission'], case['name'])
    print(f"Optimizing {case['name']} ...")
    chosen, search_diagnostic = optimize_mission(mission, case['id'])
    design = chosen['designs'][0]
    metric = chosen['mean'][0]
    design_record = {name: float(value) for name, value in zip(DESIGN_COLUMNS, design)}
    candidate_records.append({
        'case_id': case['id'], 'mission': case['mission'], 'design': design_record,
        'metrics': {
            'empty_mass_kg': float(metric[0]), 'ld': float(chosen['ld'][0]),
            'payload_volume_m3': float(metric[1]), 'fuel_volume_m3': float(metric[2]),
            'max_hotspot_stress_mpa': float(metric[3]),
        },
    })
    diagnostic_records.append({
        'case_id': case['id'], 'selected_origin': search_diagnostic['source'],
        'fallback_dataset_row': search_diagnostic['fallback_dataset_row'],
        'official_loss': float(chosen['loss'][0]), 'stress_mean_mpa': float(metric[3]),
        'stress_uncertainty_mpa': float(chosen['std'][0, 3]),
        'stress_ucb_mpa': float(chosen['stress_ucb'][0]),
        'stress_limit_mpa': float(mission['stress_limit']),
        'gate_high_probability': float(chosen['gate_probability'][0]),
        'low_expert_stress_mpa': float(chosen['low_stress'][0]),
        'high_expert_stress_mpa': float(chosen['high_stress'][0]),
        **search_diagnostic,
    })
    print(f"  {search_diagnostic['source']}: loss={chosen['loss'][0]:.5f}, mean stress={metric[3]:.2f}, UCB={chosen['stress_ucb'][0]:.2f} MPa")

candidate_payload = {'schema_version': 1, 'candidates': candidate_records}
diagnostic_payload = {
    'schema_version': 1, 'surrogate_only_validation': True,
    'stress_calibration': {'k': stress_k, 'scale_floor_mpa': stress_scale_floor},
    'model_validation': validation.to_dict(orient='records'), 'candidates': diagnostic_records,
}
candidate_path = OUTPUT_DIR / 'coupled_stress_candidates.json'
diagnostic_path = OUTPUT_DIR / 'coupled_stress_diagnostics.json'
summary_rows = []
for candidate in candidate_records:
    mission = candidate['mission']
    metrics = candidate['metrics']
    summary_rows.append({
        'case_id': candidate['case_id'],
        'Altitude (kft)': mission['altitude_kft'], 'KCAS (kt)': mission['kcas_kt'], 'AOA (deg)': mission['aoa_deg'],
        'L/D Target': mission['ld_target'],
        'Payload Volume Target (m^3)': mission['payload_volume_min_m3'],
        'Fuel Volume Target (m^3)': mission['fuel_volume_min_m3'],
        'Stress Limit (MPa)': mission['stress_max_mpa'],
        **candidate['design'],
        'Aircraft Empty Weight (kg)': metrics['empty_mass_kg'], 'L/D': metrics['ld'],
        'Payload Volume (m^3)': metrics['payload_volume_m3'],
        'Fuel Volume (m^3)': metrics['fuel_volume_m3'],
        'Max Hotspot Stress (MPa)': metrics['max_hotspot_stress_mpa'],
    })
summary_path = OUTPUT_DIR / 'coupled_stress_output_summary.csv'
pd.DataFrame(summary_rows).to_csv(summary_path, index=False)
candidate_path.write_text(json.dumps(candidate_payload, indent=2) + '\n', encoding='utf-8')
diagnostic_path.write_text(json.dumps(diagnostic_payload, indent=2) + '\n', encoding='utf-8')
print(f'Wrote {summary_path}, {candidate_path}, and {diagnostic_path}')

Optimizing High Speed Dash ...
  optimized_conservative: loss=0.69531, mean stress=41.96, UCB=334.19 MPa
Optimizing Max Endurance ...
  optimized_conservative: loss=0.51247, mean stress=41.61, UCB=333.84 MPa
Optimizing Max Capacity ...
  optimized_conservative: loss=2.03471, mean stress=42.52, UCB=334.75 MPa
Wrote /Users/vaarijbetala/Desktop/ntopwork/nTop---ASME-IDETC-CIE-Student-Hackathon/outputs/coupled_stress_output_summary.csv, /Users/vaarijbetala/Desktop/ntopwork/nTop---ASME-IDETC-CIE-Student-Hackathon/outputs/coupled_stress_candidates.json, and /Users/vaarijbetala/Desktop/ntopwork/nTop---ASME-IDETC-CIE-Student-Hackathon/outputs/coupled_stress_diagnostics.json


### Validate and visualize

This final step checks the output contract and design-variable rules, then saves a compact validation figure showing held-out stress predictions, stress-expert sampling weights, and first-restart loss histories.

In [27]:
from gauntlet.score_candidates import score_output

score_report = score_output(candidate_payload, cases_payload)
for candidate, diagnostic, case in zip(candidate_records, diagnostic_records, cases_payload['cases']):
    design = np.array([candidate['design'][name] for name in DESIGN_COLUMNS])
    assert np.all(design >= LO) and np.all(design <= HI)
    assert design[14].is_integer() and design[18].is_integer()
    assert design[17].is_integer() and int(design[17]) % 2 == 1
    if diagnostic['selected_origin'] != 'no_mean_safe_candidate':
        assert candidate['metrics']['max_hotspot_stress_mpa'] <= case['mission']['stress_max_mpa'] + 1e-9
print(f"Gauntlet contract: {'PASS' if score_report['all_passed'] else 'FAIL'}; total loss={score_report['total_loss']:.6f}")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
axes[0].scatter(stress[test_idx], test_mu[:, 3], c=test_gate, s=10, alpha=.45, cmap='viridis')
axes[0].plot([0, 1000], [0, 1000], '--', color='black', linewidth=1)
axes[0].axvline(335, color='tab:red', linestyle=':'); axes[0].axhline(335, color='tab:red', linestyle=':')
axes[0].set(xlim=(0, 1000), ylim=(0, 1000), xlabel='Observed stress (MPa)', ylabel='Coupled prediction (MPa)', title='Held-out stress prediction')
grid = np.linspace(0, 700, 500)
low_curve = 1 + 3 * gaussian_weight(grid, 335, 35) + 2 * gaussian_weight(grid, 355, 35)
high_curve = 1 + 3 * gaussian_weight(grid, 335, 30) + 3 * gaussian_weight(grid, 355, 30)
axes[1].plot(grid, low_curve, label='low expert'); axes[1].plot(grid, high_curve, label='boundary/high expert')
axes[1].axvline(335, color='tab:red', linestyle=':'); axes[1].axvline(355, color='tab:orange', linestyle=':')
axes[1].set(xlabel='Observed training stress (MPa)', ylabel='Bootstrap sampling weight', title='Dual-boundary emphasis'); axes[1].legend()
for diagnostic in diagnostic_records:
    history = diagnostic['global_searches'][0]['history']
    axes[2].plot([row['generation'] for row in history], [row['loss'] for row in history], label=diagnostic['case_id'])
axes[2].set(xlabel='DE generation', ylabel='Official loss of incumbent', title='First global restart'); axes[2].legend()
fig.tight_layout()
figure_path = OUTPUT_DIR / 'coupled_stress_validation.png'
fig.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.close(fig)
print(f'Wrote {figure_path}')

Gauntlet contract: PASS; total loss=3.242492
Wrote /Users/vaarijbetala/Desktop/ntopwork/nTop---ASME-IDETC-CIE-Student-Hackathon/outputs/coupled_stress_validation.png


## Interpretation

`optimized_conservative` means the selected off-grid design passed the calibrated one-sided stress bound. `database_mean_safe_fallback` means no optimized point passed that conservative screen, so every database geometry was mission-rescored and the lowest-loss predicted-mean-safe row was submitted. `no_mean_safe_candidate` is an explicit emergency state and never clips or fabricates stress. Perturbation agreement is evidence of basin stability, not a mathematical proof of global optimality.